# ModernBERT-large Data Preparation Pipeline for Medical Dataset Name Extraction

This notebook prepares training data for a **ModernBERT-large** token classification model that extracts dataset names from medical research articles.

It follows the same pipeline shape as `SciBERT/scibert_data_preparation.ipynb` (parse -> filter -> chunk -> negative sampling -> entity-centered augmentation -> split -> tokenize -> save), but uses the **ModernBERT-large BPE tokenizer** and exploits its **8192-token context** with larger primary chunks (6000 chars) and larger augmentation windows (4000 chars).

| Step | Description |
|------|-------------|
| 1 | Setup and configuration |
| 2 | Parse Label Studio JSON annotations |
| 3 | Filter unlabeled documents and analyze dataset |
| 4 | Entity-aware text chunking (6000 chars) |
| 5 | Controlled negative sampling (ratio 0.15) |
| 6 | Entity-centered augmentation (window 4000, max 3/doc) |
| 7 | Train/validation/test split |
| 8 | Tokenize chunks and align BIO tags to BPE tokens (max_len 8192) |
| 9 | Process Out-of-Distribution (OOD) evaluation set |
| 10 | Save all splits + metadata |

**Data balancing (same techniques as SciBERT, scaled to 8K):** negative sampling (15%) + entity-centered augmentation (4000-char windows, max 3/doc) preserve the same positive/negative balance strategy while making full use of the 8K context window.

Same 542 labeled documents and same 151-doc OOD set as CRF/SciBERT baselines. Same `SEED = 42` for deterministic chunk-level splits. Chunk counts will differ numerically from CRF/SciBERT because chunk size is 4x larger (6000 vs 1500) and augmentation windows are 5x larger (4000 vs 800).


## 1. Setup and Configuration


In [ ]:
# Install dependencies
# Uncomment the line below when running on Google Colab
%pip install -q torch transformers seqeval huggingface_hub pandas matplotlib optuna

In [4]:
import json
import os
import pickle
import random
import re
from collections import Counter
from typing import Dict, List, Optional, Tuple

from transformers import AutoTokenizer

# ============================================================
# CONFIGURATION - Modify these parameters for your experiment
# ============================================================

# --- Paths (run this notebook from MedNER/ModernBERT/) ---
DATASET_PATH = "data-annotations.json"        # Main Label Studio JSON export
OOD_DATASET_PATH = "151-eval.json"            # Out-of-distribution eval set
OUTPUT_DIR = "./modernbert_data"                 # Output directory for prepared data

# --- ModernBERT model ---
MODERNBERT_MODEL_NAME = "answerdotai/ModernBERT-large"   # Pre-trained ModernBERT-large
MAX_SEQ_LENGTH = 8192                             # ModernBERT's native context window

# --- Chunking (scaled up to use 8K context) ---
CHUNK_SIZE = 6000                                 # chars per primary chunk (was 1500 in SciBERT)
CHUNK_OVERLAP = 500                               # chars overlap between chunks (was 300)

# --- Negative sampling (KEPT unchanged from SciBERT) ---
NEGATIVE_SAMPLE_RATIO = 0.15                      # Keep 15% of entity-free chunks

# --- Augmentation (KEPT, window scaled to 8K) ---
AUGMENTATION_ENABLED = True
AUGMENTATION_WINDOW_SIZE = 4000                   # chars (was 800) - scaled 5x for 8K context
AUGMENTATION_MAX_PER_DOC = 3                      # lowered (was 5) - each example is 5x larger

# --- Split ratios (same as CRF/SciBERT pipeline) ---
TRAIN_RATIO = 0.75
VAL_RATIO = 0.10
TEST_RATIO = round(1.0 - TRAIN_RATIO - VAL_RATIO, 2)  # 0.15

# --- Reproducibility (same seed as CRF/SciBERT) ---
SEED = 42

# rng uses the same seed as upstream pipelines - chunk counts differ because chunk size
# changed from 1500 to 6000 chars, but the RNG sequence is deterministic
rng = random.Random(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Label mapping ---
label2id = {"O": 0, "B-Dataset": 1, "I-Dataset": 2}
id2label = {v: k for k, v in label2id.items()}

# --- Load ModernBERT tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODERNBERT_MODEL_NAME, use_fast=True)
assert tokenizer.is_fast, "Expected a fast (Rust-based) tokenizer for word_ids() support"

print(f"ModernBERT model:    {MODERNBERT_MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Output directory:    {OUTPUT_DIR}")
print(f"Chunk size:          {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP} chars")
print(f"Negative sample ratio: {NEGATIVE_SAMPLE_RATIO}")
print(f"Augmentation:        {'ON' if AUGMENTATION_ENABLED else 'OFF'} "
      f"(window={AUGMENTATION_WINDOW_SIZE}, max_per_doc={AUGMENTATION_MAX_PER_DOC})")
print(f"Split:               {TRAIN_RATIO:.0%} train / {VAL_RATIO:.0%} val / {TEST_RATIO:.0%} test")
print(f"Seed:                {SEED}")
print(f"Label mapping:       {label2id}")
print(f"OOD eval set:        {OOD_DATASET_PATH}")
print(f"Tokenizer fast:      {tokenizer.is_fast}")


ModernBERT model:    answerdotai/ModernBERT-large
Max sequence length: 8192
Output directory:    ./modernbert_data
Chunk size:          6000 chars, overlap: 500 chars
Negative sample ratio: 0.15
Augmentation:        ON (window=4000, max_per_doc=3)
Split:               75% train / 10% val / 15% test
Seed:                42
Label mapping:       {'O': 0, 'B-Dataset': 1, 'I-Dataset': 2}
OOD eval set:        151-eval.json
Tokenizer fast:      True


## 2. Parse Label Studio Data

The dataset is in **Label Studio JSON** format. Each item contains:
- `data.text` - the full document text
- `annotations[].result[]` - list of annotation results, each with:
  - `value.start` / `value.end` - character offsets
  - `value.text` - the annotated span text
  - `value.labels` - list of entity type labels (e.g., `["Dataset"]`)


The `parse_label_studio_json` function below is identical to the one used in the CRF baseline notebook. It normalizes whitespace (replaces `
`, `
`, `	` with spaces) while preserving character offsets, then collects all annotated entity spans.

In [5]:
def parse_label_studio_json(filepath: str) -> List[Dict]:
    """
    Parse a Label Studio JSON export into a list of documents.
    Normalizes document text (replaces newlines/tabs with spaces).
    
    Returns list of dicts:
        {'text': str, 'entities': [{'start', 'end', 'text', 'label'}], 'is_labeled': bool}
    """
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    parsed = []
    for item in data:
        raw_text = (item.get("data") or {}).get("text", "")
        if not raw_text:
            continue

        # Normalize: replace newlines/tabs with spaces (preserves char offsets)
        text = raw_text.replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

        entities = []
        for annot in item.get("annotations") or []:
            for r in annot.get("result") or []:
                value = r.get("value") or {}
                start = value.get("start")
                end = value.get("end")
                labels = value.get("labels") or []
                if start is not None and end is not None:
                    entity_text = text[start:end].strip()
                    if not entity_text:
                        continue
                    # Adjust offsets to match stripped text
                    offset = text[start:end].index(entity_text) if entity_text in text[start:end] else 0
                    adj_start = start + offset
                    adj_end = adj_start + len(entity_text)
                    for label in labels:
                        entities.append(
                            {"start": adj_start, "end": adj_end, "text": entity_text, "label": label}
                        )

        parsed.append({"text": text, "entities": entities, "is_labeled": len(entities) > 0})

    return parsed


In [6]:
documents = parse_label_studio_json(DATASET_PATH)

n_labeled = sum(1 for d in documents if d["is_labeled"])
n_unlabeled = sum(1 for d in documents if not d["is_labeled"])

print(f"Total documents: {len(documents)}")
print(f"  Labeled:   {n_labeled}")
print(f"  Unlabeled: {n_unlabeled}")


Total documents: 954
  Labeled:   542
  Unlabeled: 412


In [7]:
# Display 5 sample documents
print("=" * 60)
print("SAMPLE DOCUMENTS")
print("=" * 60)

for i, doc in enumerate(documents[:5]):
    print(f"\n--- Document {i+1} ---")
    print(f"Text length: {len(doc['text'])} chars")
    print(f"Text preview: {doc['text'][:200]}...")
    print(f"Entities ({len(doc['entities'])}):")
    for ent in doc["entities"][:10]:
        print(f"  [{ent['start']}:{ent['end']}] {ent['label']}: \"{ent['text']}\"")
    if len(doc["entities"]) > 10:
        print(f"  ... and {len(doc['entities']) - 10} more")


SAMPLE DOCUMENTS

--- Document 1 ---
Text length: 25176 chars
Text preview: Annotating Synapses in Large EM Datasets Reconstructing neuronal circuits at the level of synapses is a central problem in neuroscience and becoming a focus of the emerging field of connectomics. To d...
Entities (4):
  [977:993] Dataset: "EM image dataset"
  [2082:2096] Dataset: "FIB-SEM images"
  [2082:2096] Dataset: "FIB-SEM images"
  [980:994] Dataset: "image dataset."

--- Document 2 ---
Text length: 10302 chars
Text preview: Automatic Brain Tumor Segmentation with Scale Attention Network Automatic segmentation of brain tumors is an essential but challenging step for extracting quantitative imaging biomarkers for accurate ...
Entities (6):
  [8118:8128] Dataset: "BraTS 2020"
  [9214:9224] Dataset: "BraTS 2020"
  [9388:9398] Dataset: "Brats 2020"
  [274:337] Dataset: "Multimodal Brain Tumor Segmentation Challenge 2020 (BraTS 2020)"
  [1154:1164] Dataset: "BraTS 2020"
  [1486:1496] Dataset: "BraTS 2020"

---

## 3. Filter & Analyze


In [8]:
# Filter unlabeled documents
labeled_docs = [doc for doc in documents if doc["is_labeled"]]
n_removed = len(documents) - len(labeled_docs)

print(f"Filtered out {n_removed} unlabeled documents.")
print(f"Remaining labeled documents: {len(labeled_docs)}")

# Dataset statistics
label_counter = Counter()
entities_per_doc = []
text_lengths = []
entity_text_lengths = []

for doc in labeled_docs:
    text_lengths.append(len(doc["text"]))
    entities_per_doc.append(len(doc["entities"]))
    for ent in doc["entities"]:
        label_counter[ent["label"]] += 1
        entity_text_lengths.append(len(ent["text"]))

print("="  * 60)
print("DATASET STATISTICS")
print("=" * 60)
print(f"Documents:           {len(labeled_docs)}")
print(f"Total entities:      {sum(label_counter.values())}")
print(f"Label types:         {dict(label_counter)}")
print(f"Entities per doc:    mean={sum(entities_per_doc)/len(entities_per_doc):.1f}, "
      f"min={min(entities_per_doc)}, max={max(entities_per_doc)}")
print(f"Text length (chars): mean={sum(text_lengths)/len(text_lengths):.0f}, "
      f"min={min(text_lengths)}, max={max(text_lengths)}")
print(f"Entity text length:  mean={sum(entity_text_lengths)/len(entity_text_lengths):.1f}, "
      f"min={min(entity_text_lengths)}, max={max(entity_text_lengths)}")


Filtered out 412 unlabeled documents.
Remaining labeled documents: 542
DATASET STATISTICS
Documents:           542
Total entities:      6353
Label types:         {'Dataset': 6353}
Entities per doc:    mean=11.7, min=1, max=109
Text length (chars): mean=26383, min=4065, max=299869
Entity text length:  mean=13.9, min=2, max=194


## 4. Entity-Aware Text Chunking

Documents average ~26K characters. We use a sliding window approach scaled to ModernBERT's 8K context:
- **Window size**: 6000 characters (~1800 BPE tokens - plenty of headroom under 8192)
- **Overlap**: 500 characters to avoid splitting entities at boundaries
- **Sentence-boundary alignment**: tries to break at sentence ends
- **Entity mapping**: only includes entities fully contained within a chunk

Compared to SciBERT's 1500-char chunks, these are 4x larger - expect roughly 1/4 the chunk count from the same document pool.


In [9]:
def chunk_document(doc: Dict, chunk_size: int, chunk_overlap: int) -> List[Dict]:
    """
    Split a long document into overlapping chunks.
    Maps entity spans to their new positions within each chunk.
    Entities spanning chunk boundaries are discarded.
    """
    text = doc["text"]
    entities = doc["entities"]
    text_len = len(text)
    chunks = []
    start = 0

    while start < text_len:
        end = min(start + chunk_size, text_len)
        if end < text_len:
            search_start = max(start, end - 200)
            for sep in [". ", ".\n", "\n\n", "\n", " "]:
                last_sep = text[search_start:end].rfind(sep)
                if last_sep != -1:
                    end = search_start + last_sep + len(sep)
                    break
        chunk_text = text[start:end]
        chunk_entities = []
        for ent in entities:
            if ent["start"] >= start and ent["end"] <= end:
                chunk_entities.append({
                    "start": ent["start"] - start,
                    "end": ent["end"] - start,
                    "text": ent["text"],
                    "label": ent["label"],
                })
        chunks.append({
            "text": chunk_text,
            "entities": chunk_entities,
            "has_entities": len(chunk_entities) > 0,
            "source": "chunk",
        })
        if end >= text_len:
            break
        start = end - chunk_overlap
    return chunks


def chunk_all_documents(documents: List[Dict]) -> List[Dict]:
    """Chunk all documents and return flat list of chunks."""
    all_chunks = []
    for doc in documents:
        all_chunks.extend(chunk_document(doc, CHUNK_SIZE, CHUNK_OVERLAP))
    return all_chunks

In [10]:
all_chunks = chunk_all_documents(labeled_docs)

n_pos = sum(1 for c in all_chunks if c["has_entities"])
n_neg = sum(1 for c in all_chunks if not c["has_entities"])
total_ents = sum(len(c["entities"]) for c in all_chunks)

print(f"Total primary chunks: {len(all_chunks)}")
print(f"  With entities:    {n_pos} ({n_pos/len(all_chunks):.1%})")
print(f"  Without entities: {n_neg} ({n_neg/len(all_chunks):.1%})")
print(f"  Total entity mentions: {total_ents}")
print(f"\n(SciBERT baseline had ~12730 chunks at 1500 chars; expect ~1/4 of that at 6000 chars.)")


Total primary chunks: 2862
  With entities:    1570 (54.9%)
  Without entities: 1292 (45.1%)
  Total entity mentions: 6912

(SciBERT baseline had ~12730 chunks at 1500 chars; expect ~1/4 of that at 6000 chars.)


## 5. Controlled Negative Sampling

**Kept identical to SciBERT:** keep ALL entity-containing chunks, subsample entity-free chunks at 15% ratio. Even at 6000-char primary chunks some regions (especially in long OOD documents) are entity-free; this keeps the positive/negative balance consistent with the SciBERT/CRF runs.


In [11]:
def apply_negative_sampling(chunks: List[Dict], ratio: float) -> Tuple[List[Dict], Dict]:
    """Keep all entity-containing chunks. Subsample entity-free chunks at the given ratio."""
    positive = [c for c in chunks if c["has_entities"]]
    negative = [c for c in chunks if not c["has_entities"]]
    n_neg_keep = max(1, int(len(negative) * ratio))
    sampled_neg = rng.sample(negative, min(n_neg_keep, len(negative)))
    balanced = positive + sampled_neg
    rng.shuffle(balanced)
    stats = {
        "positive_chunks": len(positive),
        "total_negative_chunks": len(negative),
        "sampled_negative_chunks": len(sampled_neg),
        "total_after_sampling": len(balanced),
    }
    return balanced, stats


balanced_chunks, neg_stats = apply_negative_sampling(all_chunks, NEGATIVE_SAMPLE_RATIO)

print(f"Negative Sampling Results:")
print(f"  Positive chunks (kept all):     {neg_stats['positive_chunks']}")
print(f"  Negative chunks (before):       {neg_stats['total_negative_chunks']}")
print(f"  Negative chunks (after):        {neg_stats['sampled_negative_chunks']}")
print(f"  Total after sampling:           {neg_stats['total_after_sampling']}")

Negative Sampling Results:
  Positive chunks (kept all):     1570
  Negative chunks (before):       1292
  Negative chunks (after):        193
  Total after sampling:           1763


## 6. Entity-Centered Augmentation

**Kept from SciBERT, window scaled 5x for 8K context:** create additional focused windows (4000 chars, up from 800) centered on entity mentions. `MAX_PER_DOC` lowered from 5 to 3 because each augmented example is now 5x larger in memory. This oversamples entity-rich regions while staying well under the 8192-token limit.


In [12]:
def create_entity_centered_chunks(doc: Dict) -> List[Dict]:
    """Create smaller, focused chunks centered on entity mentions."""
    if not doc["entities"]:
        return []
    text = doc["text"]
    text_len = len(text)
    half_window = AUGMENTATION_WINDOW_SIZE // 2
    aug_chunks = []
    used_centers = set()
    shuffled_entities = list(doc["entities"])
    rng.shuffle(shuffled_entities)
    for ent in shuffled_entities:
        if len(aug_chunks) >= AUGMENTATION_MAX_PER_DOC:
            break
        center = (ent["start"] + ent["end"]) // 2
        if any(abs(center - uc) < half_window // 2 for uc in used_centers):
            continue
        used_centers.add(center)
        win_start = max(0, center - half_window)
        win_end = min(text_len, center + half_window)
        for sep in [". ", "\n"]:
            idx = text[win_start:min(win_start + 100, center)].find(sep)
            if idx != -1:
                win_start = win_start + idx + len(sep)
                break
        chunk_text = text[win_start:win_end]
        chunk_entities = []
        for e in doc["entities"]:
            if e["start"] >= win_start and e["end"] <= win_end:
                chunk_entities.append({
                    "start": e["start"] - win_start,
                    "end": e["end"] - win_start,
                    "text": e["text"],
                    "label": e["label"],
                })
        if chunk_entities:
            aug_chunks.append({
                "text": chunk_text,
                "entities": chunk_entities,
                "has_entities": True,
                "source": "augmentation",
            })
    return aug_chunks


if AUGMENTATION_ENABLED:
    aug_chunks = []
    for doc in labeled_docs:
        aug_chunks.extend(create_entity_centered_chunks(doc))
    print(f"Entity-centered augmentation chunks created: {len(aug_chunks)}")
    print(f"  Avg entities per aug chunk: {sum(len(c['entities']) for c in aug_chunks)/max(len(aug_chunks),1):.2f}")
    combined_chunks = balanced_chunks + aug_chunks
    rng.shuffle(combined_chunks)
    print(f"\nTotal chunks after augmentation: {len(combined_chunks)}")
    print(f"  From regular chunking: {len(balanced_chunks)}")
    print(f"  From augmentation:     {len(aug_chunks)}")
else:
    combined_chunks = balanced_chunks
    print("Augmentation disabled. Using balanced chunks only.")
    print(f"Total chunks: {len(combined_chunks)}")

Entity-centered augmentation chunks created: 1336
  Avg entities per aug chunk: 4.20

Total chunks after augmentation: 3099
  From regular chunking: 1763
  From augmentation:     1336


## 7. Train/Validation/Test Split

Split at the **chunk level** before ModernBERT tokenization, using the same `SEED = 42` RNG as CRF/SciBERT. Splits are deterministic but chunk *counts* differ numerically from CRF/SciBERT because chunk size (1500 -> 6000) and augmentation window (800 -> 4000) both scaled up.


In [13]:
# Split chunks directly (before ModernBERT tokenization)
chunks_to_split = list(combined_chunks)
rng.shuffle(chunks_to_split)

n_total = len(chunks_to_split)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)

train_chunks = chunks_to_split[:n_train]
val_chunks = chunks_to_split[n_train:n_train + n_val]
test_chunks = chunks_to_split[n_train + n_val:]

print(f"Split results:")
print(f"{'Split':<12} {'Chunks':>8} {'Entities':>10}")
print("-" * 35)
for name, chunks in [("Train", train_chunks), ("Val", val_chunks), ("Test", test_chunks)]:
    n_ents = sum(len(c['entities']) for c in chunks)
    print(f"{name:<12} {len(chunks):>8} {n_ents:>10}")
total = len(train_chunks) + len(val_chunks) + len(test_chunks)
print("-" * 35)
print(f"{'Total':<12} {total:>8}")

print(f"\nNote: chunk counts will not match SciBERT/CRF (train=5133, val=684, test=1028)")
print(f"because ModernBERT uses 6000-char primary chunks and 4000-char augmentation windows.")
print(f"The underlying 542-document pool and seed are identical, so comparison stays valid at the entity-set / F1 level.")


Split results:
Split          Chunks   Entities
-----------------------------------
Train            2324       9236
Val               309       1341
Test              466       1951
-----------------------------------
Total            3099

Note: chunk counts will not match SciBERT/CRF (train=5133, val=684, test=1028)
because ModernBERT uses 6000-char primary chunks and 4000-char augmentation windows.
The underlying 542-document pool and seed are identical, so comparison stays valid at the entity-set / F1 level.


## 8. Convert Chunks to ModernBERT BIO-Tagged Sequences

Each chunk is tokenized with the ModernBERT BPE tokenizer (max_length=8192). Entity character offsets are aligned to subword tokens using `word_ids()` from the fast tokenizer:

- **First subword** of each word gets the BIO label (B-Dataset, I-Dataset, or O)
- **Subsequent subwords** of the same word get -100 (ignored by loss function)
- **Special tokens** (CLS/SEP equivalents) get -100
- **Padding** positions get -100

`CrossEntropyLoss(ignore_index=-100)` automatically skips all non-first-subword positions during training.

The alignment logic is tokenizer-agnostic (WordPiece, BPE, or SentencePiece) as long as the tokenizer is fast and provides `word_ids()` and `offset_mapping`.


In [14]:
def align_entities_to_tokens(entities, offset_mapping, word_ids_raw, max_length):
    """
    Align entity character offsets to subword tokens produced by a fast tokenizer.
    Tokenizer-agnostic (works with WordPiece, BPE, SentencePiece).

    Args:
        entities: list of {'start': int, 'end': int, 'label': str}
        offset_mapping: list of (char_start, char_end) per token from tokenizer
        word_ids_raw: list from encoding.word_ids() -- word index or None for special tokens
        max_length: max sequence length

    Returns:
        labels: List[int] -- BIO label per token (-100 for special/subword tokens)
        first_subword_mask: List[bool] -- True for first-subword tokens of each word
    """
    num_tokens = len(offset_mapping)
    labels = [-100] * num_tokens
    first_subword_mask = [False] * num_tokens

    # Mark first-subword tokens with default O label
    seen_words = set()
    for i in range(num_tokens):
        wid = word_ids_raw[i]
        if wid is not None and wid not in seen_words:
            seen_words.add(wid)
            labels[i] = 0  # O
            first_subword_mask[i] = True

    # Assign entity BIO tags using character offsets
    for ent in entities:
        ent_start, ent_end = ent['start'], ent['end']
        first_word_of_entity = True
        tagged_words = set()

        for i in range(num_tokens):
            tok_start, tok_end = offset_mapping[i]
            wid = word_ids_raw[i]

            if wid is None or tok_end == 0:       # special token
                continue
            if tok_start >= ent_end or tok_end <= ent_start:  # no overlap
                continue
            if wid in tagged_words:                # already tagged this word
                continue
            if not first_subword_mask[i]:          # not a first-subword token
                continue

            tagged_words.add(wid)
            if first_word_of_entity:
                labels[i] = 1  # B-Dataset
                first_word_of_entity = False
            else:
                labels[i] = 2  # I-Dataset

    return labels, first_subword_mask


In [15]:
def chunk_to_modernbert_bio(chunk, tokenizer, max_length=8192):
    """
    Convert a chunk dict to a ModernBERT BIO-labeled sequence.

    Returns dict with input_ids, attention_mask, labels, first_subword_mask,
    word_ids, gold_entities, offset_mapping, text, and truncation flag.
    """
    text = chunk['text']
    entities = chunk.get('entities', [])

    # Tokenize with offset mapping
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        return_attention_mask=True,
    )

    input_ids = encoding['input_ids']
    attention_mask = encoding['attention_mask']
    offset_mapping = encoding['offset_mapping']
    word_ids_raw = encoding.word_ids()

    # Check truncation
    full_length = len(tokenizer.encode(text, add_special_tokens=True))
    was_truncated = full_length > max_length

    # Filter entities that fall beyond truncation boundary
    if was_truncated and offset_mapping:
        last_real_offset = max(
            (om[1] for om in offset_mapping if om != (0, 0)),
            default=0
        )
        valid_entities = [e for e in entities if e['end'] <= last_real_offset]
    else:
        valid_entities = entities

    # Align entities to tokens
    labels, first_subword_mask = align_entities_to_tokens(
        valid_entities, offset_mapping, word_ids_raw, max_length
    )

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
        # Kept under the legacy key 'crf_mask' for compatibility with evaluation helpers
        # that iterate over first-subword positions. It's the same boolean mask.
        'crf_mask': first_subword_mask,
        'word_ids': word_ids_raw,
        'offset_mapping': offset_mapping,
        'gold_entities': [e['text'] for e in valid_entities],
        'was_truncated': was_truncated,
        'text': text,
    }


def convert_all_chunks(chunks, tokenizer, max_length, split_name=""):
    """Convert a list of chunks to ModernBERT BIO sequences."""
    examples = []
    truncated = 0
    for chunk in chunks:
        ex = chunk_to_modernbert_bio(chunk, tokenizer, max_length)
        examples.append(ex)
        if ex['was_truncated']:
            truncated += 1

    print(f"{split_name} conversion: {len(examples)} examples, "
          f"{truncated} truncated ({truncated/max(len(examples),1):.1%})")
    return examples


In [16]:
# Convert all splits to ModernBERT BIO sequences
train_examples = convert_all_chunks(train_chunks, tokenizer, MAX_SEQ_LENGTH, "Train")
val_examples = convert_all_chunks(val_chunks, tokenizer, MAX_SEQ_LENGTH, "Val")
test_examples = convert_all_chunks(test_chunks, tokenizer, MAX_SEQ_LENGTH, "Test")

# Sanity check: at 8K max_length with 6000-char primary chunks and 4000-char augmentation
# windows, truncation should be essentially 0% for primary+augmented chunks.
train_max_tokens = max(len(ex['input_ids']) for ex in train_examples) if train_examples else 0
print(f"\nTrain max token length observed: {train_max_tokens} (limit={MAX_SEQ_LENGTH})")


Train conversion: 2324 examples, 0 truncated (0.0%)
Val conversion: 309 examples, 0 truncated (0.0%)
Test conversion: 466 examples, 0 truncated (0.0%)

Train max token length observed: 2774 (limit=8192)


In [17]:
# Token length distribution and BIO tag statistics
print("=" * 60)
print("MODERNBERT TOKENIZATION STATISTICS")
print("=" * 60)

for name, examples in [("Train", train_examples), ("Val", val_examples), ("Test", test_examples)]:
    lengths = [len(ex['input_ids']) for ex in examples]
    n_active = [sum(1 for l in ex['labels'] if l != -100) for ex in examples]

    # BIO tag distribution
    tag_counts = Counter()
    for ex in examples:
        for label in ex['labels']:
            if label != -100:
                tag_counts[id2label[label]] += 1

    total_tags = sum(tag_counts.values())
    print(f"{name} ({len(examples)} examples):")
    print(f"  Token lengths: mean={sum(lengths)/len(lengths):.0f}, "
          f"min={min(lengths)}, max={max(lengths)}")
    print(f"  Active tokens (labels != -100): mean={sum(n_active)/len(n_active):.0f}")
    print(f"  BIO distribution:")
    for tag in ["O", "B-Dataset", "I-Dataset"]:
        count = tag_counts.get(tag, 0)
        print(f"    {tag:<15} {count:>8} ({count/total_tags:.2%})")

# Display 5 sample sequences with entities
print("" + "=" * 60)
print("SAMPLE MODERNBERT BIO-TAGGED SEQUENCES")
print("=" * 60)

sample_count = 0
for ex in train_examples:
    if any(l > 0 for l in ex['labels'] if l != -100):
        print(f"--- Sample {sample_count + 1} ---")
        print(f"Text: {ex['text'][:100]}...")
        print(f"Gold entities: {ex['gold_entities']}")
        n_active = sum(1 for l in ex['labels'] if l != -100)
        print(f"Tokens: {len(ex['input_ids'])}, Active: {n_active}")
        print(f"{'Token':<20} {'WordID':<8} {'Label'}")
        print("-" * 40)
        tokens = tokenizer.convert_ids_to_tokens(ex['input_ids'])
        for j in range(min(len(tokens), 40)):
            tok = tokens[j]
            wid = ex['word_ids'][j]
            label = ex['labels'][j]
            label_str = id2label[label] if label != -100 else "-100"
            marker = " <<" if label > 0 else ""
            wid_str = str(wid) if wid is not None else "-"
            print(f"{tok:<20} {wid_str:<8} {label_str}{marker}")
        if len(tokens) > 40:
            print(f"  ... ({len(tokens) - 40} more tokens)")
        sample_count += 1
        if sample_count >= 5:
            break


MODERNBERT TOKENIZATION STATISTICS
Train (2324 examples):
  Token lengths: mean=1027, min=98, max=2774
  Active tokens (labels != -100): mean=939
  BIO distribution:
    O                2156197 (98.83%)
    B-Dataset           9161 (0.42%)
    I-Dataset          16445 (0.75%)
Val (309 examples):
  Token lengths: mean=1014, min=123, max=2400
  Active tokens (labels != -100): mean=929
  BIO distribution:
    O                 283279 (98.66%)
    B-Dataset           1325 (0.46%)
    I-Dataset           2509 (0.87%)
Test (466 examples):
  Token lengths: mean=1033, min=123, max=2781
  Active tokens (labels != -100): mean=948
  BIO distribution:
    O                 436292 (98.74%)
    B-Dataset           1949 (0.44%)
    I-Dataset           3602 (0.82%)
SAMPLE MODERNBERT BIO-TAGGED SEQUENCES
--- Sample 1 ---
Text: he Rotation Eq. CNN (0.963) and Densenet models (0.962) on the PCam dataset with a drastic reduction...
Gold entities: ['PCam dataset', 'PCam dataset', 'OASIS dataset']
Tokens: 

## 9. Process Out-of-Distribution (OOD) Evaluation Set

The OOD set (`../151-eval.json`) is processed with the **same full pipeline** as the main dataset:
- **Same chunking** -- entity-aware sliding window (6000 chars, 500 overlap)
- **Same negative sampling** -- 15% ratio for consistency with val/test splits
- **Same augmentation** -- entity-centered chunks (4000-char windows, max 3/doc) for balanced entity representation
- **No split** -- entire set used for evaluation

This ensures fair within-model comparison across val, test, and OOD splits, and mirrors what SciBERT/CRF did on their OOD runs.


In [18]:
# Parse OOD evaluation set
ood_docs = parse_label_studio_json(OOD_DATASET_PATH)

n_ood_labeled = sum(1 for d in ood_docs if d['is_labeled'])
n_ood_unlabeled = sum(1 for d in ood_docs if not d['is_labeled'])
total_ood_ents = sum(len(d['entities']) for d in ood_docs)

print(f"OOD evaluation set: {OOD_DATASET_PATH}")
print(f"  Total documents: {len(ood_docs)}")
print(f"  Labeled:   {n_ood_labeled}")
print(f"  Unlabeled: {n_ood_unlabeled}")
print(f"  Total entity annotations: {total_ood_ents}")

# Display 5 sample OOD documents
print("\n" + "=" * 60)
print("SAMPLE OOD DOCUMENTS")
print("=" * 60)
for i, doc in enumerate(ood_docs[:5]):
    print(f"\n--- OOD Document {i+1} ---")
    print(f"Text length: {len(doc['text'])} chars")
    print(f"Text preview: {doc['text'][:150]}...")
    print(f"Entities ({len(doc['entities'])}):")
    for ent in doc['entities'][:5]:
        print(f"  [{ent['start']}:{ent['end']}] {ent['label']}: \"{ent['text']}\"")
    if len(doc['entities']) > 5:
        print(f"  ... and {len(doc['entities']) - 5} more")

OOD evaluation set: 151-eval.json
  Total documents: 151
  Labeled:   151
  Unlabeled: 0
  Total entity annotations: 3086

SAMPLE OOD DOCUMENTS

--- OOD Document 1 ---
Text length: 34312 chars
Text preview: A Feasibility Study of Literature-Guided HRV Stratification Using Large Language Models Background: Heart rate variability (HRV) is a valuable indicat...
Entities (1):
  [2825:2850] Dataset: "PhysioNet SHAREE database"

--- OOD Document 2 ---
Text length: 23748 chars
Text preview: A machine-learning informed circulating microbial DNA signature for early diagnosis of esophageal adenocarcinoma Esophageal adenocarcinoma (EAC) has s...
Entities (1):
  [5577:5604] Dataset: "k2_pluspf_20210517 database"

--- OOD Document 3 ---
Text length: 39435 chars
Text preview: A microenvironment-determined risk continuum refines subtyping in meningioma and reveals determinants of machine learning-based tumor classification C...
Entities (14):
  [13066:13095] Dataset: "Gene Expression Omnibus (GEO)"
 

In [19]:
# Chunk OOD documents
ood_docs_labeled = [doc for doc in ood_docs if doc['is_labeled']]
ood_all_chunks = chunk_all_documents(ood_docs_labeled)

ood_n_pos = sum(1 for c in ood_all_chunks if c['has_entities'])
ood_n_neg = sum(1 for c in ood_all_chunks if not c['has_entities'])

print(f"OOD raw primary chunks: {len(ood_all_chunks)}")
print(f"  With entities:    {ood_n_pos} ({ood_n_pos/len(ood_all_chunks):.1%})")
print(f"  Without entities: {ood_n_neg} ({ood_n_neg/len(ood_all_chunks):.1%})")

# Apply same negative sampling as main set
ood_balanced_chunks, ood_neg_stats = apply_negative_sampling(ood_all_chunks, NEGATIVE_SAMPLE_RATIO)

print(f"\nOOD Negative Sampling:")
print(f"  Positive chunks (kept all):     {ood_neg_stats['positive_chunks']}")
print(f"  Negative chunks (before):       {ood_neg_stats['total_negative_chunks']}")
print(f"  Negative chunks (after):        {ood_neg_stats['sampled_negative_chunks']}")
print(f"  Total after sampling:           {ood_neg_stats['total_after_sampling']}")

# Apply same entity-centered augmentation as main set (4000-char windows, max 3/doc)
if AUGMENTATION_ENABLED:
    ood_aug_chunks = []
    for doc in ood_docs_labeled:
        ood_aug_chunks.extend(create_entity_centered_chunks(doc))

    print(f"\nOOD Entity-Centered Augmentation:")
    print(f"  Augmentation chunks created: {len(ood_aug_chunks)}")

    ood_combined_chunks = ood_balanced_chunks + ood_aug_chunks
    rng.shuffle(ood_combined_chunks)

    print(f"\nOOD Total chunks after augmentation: {len(ood_combined_chunks)}")
    print(f"  From regular chunking: {len(ood_balanced_chunks)}")
    print(f"  From augmentation:     {len(ood_aug_chunks)}")
else:
    ood_combined_chunks = ood_balanced_chunks
    print(f"\nAugmentation disabled. OOD chunks: {len(ood_combined_chunks)}")

# Convert OOD to ModernBERT BIO
ood_examples = convert_all_chunks(ood_combined_chunks, tokenizer, MAX_SEQ_LENGTH, "OOD")


OOD raw primary chunks: 1339
  With entities:    667 (49.8%)
  Without entities: 672 (50.2%)

OOD Negative Sampling:
  Positive chunks (kept all):     667
  Negative chunks (before):       672
  Negative chunks (after):        100
  Total after sampling:           767

OOD Entity-Centered Augmentation:
  Augmentation chunks created: 375

OOD Total chunks after augmentation: 1142
  From regular chunking: 767
  From augmentation:     375
OOD conversion: 1142 examples, 0 truncated (0.0%)


## 10. Save All Data

In [20]:
# Save all splits as pickle files
for name, examples in [("train", train_examples), ("val", val_examples),
                        ("test", test_examples), ("ood", ood_examples)]:
    path = os.path.join(OUTPUT_DIR, f"{name}.pkl")
    with open(path, 'wb') as f:
        pickle.dump(examples, f)
    print(f"Saved {name}.pkl ({len(examples)} examples, {os.path.getsize(path)/1024:.1f} KB)")

# Save metadata
metadata = {
    "source_file": DATASET_PATH,
    "ood_source_file": OOD_DATASET_PATH,
    "modernbert_model": MODERNBERT_MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "tokenizer": "modernbert_bpe",
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "negative_sampling": True,
    "negative_sample_ratio": NEGATIVE_SAMPLE_RATIO,
    "augmentation": AUGMENTATION_ENABLED,
    "augmentation_window_size": AUGMENTATION_WINDOW_SIZE,
    "augmentation_max_per_doc": AUGMENTATION_MAX_PER_DOC,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "seed": SEED,
    "bio_labels": ["O", "B-Dataset", "I-Dataset"],
    "label2id": label2id,
    "splits": {
        "train": {"examples": len(train_examples)},
        "val":   {"examples": len(val_examples)},
        "test":  {"examples": len(test_examples)},
        "ood":   {"examples": len(ood_examples)},
    },
    "ood_processing": {
        "negative_sampling": True,
        "augmentation": AUGMENTATION_ENABLED,
        "note": "Same pipeline as main set for consistent comparison",
    },
    "total_documents": len(labeled_docs),
    "ood_documents": len(ood_docs_labeled),
    "comparison_note": (
        "Same pipeline shape as SciBERT (neg sampling + augmentation), "
        "but chunk size scaled 1500->6000 and augmentation window 800->4000 "
        "to exploit 8K context. Chunk counts therefore differ from CRF/SciBERT."
    ),
}

meta_path = os.path.join(OUTPUT_DIR, "metadata.json")
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print(f"\nSaved metadata.json ({os.path.getsize(meta_path)/1024:.1f} KB)")


Saved train.pkl (2324 examples, 54573.5 KB)
Saved val.pkl (309 examples, 7170.8 KB)
Saved test.pkl (466 examples, 11033.4 KB)
Saved ood.pkl (1142 examples, 27282.1 KB)

Saved metadata.json (1.2 KB)


In [21]:
# Final summary
print("=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)
print(f"\nSource: {DATASET_PATH} ({len(labeled_docs)} labeled documents)")
print(f"OOD:    {OOD_DATASET_PATH} ({len(ood_docs_labeled)} labeled documents)")
print(f"Output: {OUTPUT_DIR}/")
print(f"\nFiles generated:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    print(f"  {fname:<30} {os.path.getsize(fpath)/1024:>8.1f} KB")
print(f"\nSplit summary:")
print(f"  Train: {len(train_examples)} examples")
print(f"  Val:   {len(val_examples)} examples")
print(f"  Test:  {len(test_examples)} examples")
print(f"  OOD:   {len(ood_examples)} examples")
print(f"\nProcessing: All splits use same pipeline (chunking + neg sampling + augmentation)")
print(f"Tokenizer: ModernBERT BPE ({MODERNBERT_MODEL_NAME})")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"\nReady for ModernBERT-large training with modernbert_model.ipynb")


PIPELINE COMPLETE

Source: data-annotations.json (542 labeled documents)
OOD:    151-eval.json (151 labeled documents)
Output: ./modernbert_data/

Files generated:
  metadata.json                       1.2 KB
  ood.pkl                         27282.1 KB
  test.pkl                        11033.4 KB
  train.pkl                       54573.5 KB
  val.pkl                          7170.8 KB

Split summary:
  Train: 2324 examples
  Val:   309 examples
  Test:  466 examples
  OOD:   1142 examples

Processing: All splits use same pipeline (chunking + neg sampling + augmentation)
Tokenizer: ModernBERT BPE (answerdotai/ModernBERT-large)
Max sequence length: 8192

Ready for ModernBERT-large training with modernbert_model.ipynb
